## Analysis extension #1 
### Revision of the Original Pipeline

### Scope boundary
AE1 documents the original pipeline and identifies some revision issues. It does **not** rebuild the developmental hierarchy, add endocrine directions, perform K-sensitivity/null models, retrain the MLM, or run PCA/scVI controls.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## 1. Summary

### Developmental training set
The Qiu *et al.* prenatal mouse atlas was used as a broad endodermal/epithelial reference rather than a pancreas-only subset (`STRICT_PANCREAS_ONLY = False`). Coarse developmental states were balanced to **at most 4,000 cells per state** with **seed 13**, yielding **34,760 developmental cells**.

### Cancer training set
The Burdziak progression cohort contained **28,131 cells**. An epithelial-expression score was explored, but the thresholded 19,692-cell subset was **not used**. The full 28,131-cell cohort was retained. The source study had already experimentally enriched the cohort for lineage-traced pancreatic epithelial cells by FACS.

### Joint MLM corpus
- Development: **34,760**
- Cancer: **28,131**
- Total: **62,891 cells**
- Shared gene vocabulary: **15,376 Ensembl genes**

Development and cancer were trained **jointly** in one BERT model using dataset-domain tokens `[DEV]` and `[CANCER]`.

### Main audit findings
1. **85 conjunctival goblet cells** entered the developmental corpus through an overly broad annotation rule. Downstream sensitivity testing showed negligible influence.
2. The downstream analysis used `foregut_other` (thyroid/thymic/parathyroid epithelium) as the common “foregut” root, whereas actual Qiu `Foregut epithelial cells` were mapped separately as a ductal proxy. This is a substantive downstream biological-definition issue to be fixed in AE3.
3. Four duplicate Ensembl mappings were collapsed by keeping the first occurrence despite a `duplicate_agg="sum"` argument. This is an internal reproducibility/code-cleanup note for the definitive rerun.

In [ ]:
# ORIGINAL PROJECT PATH USED IN THE SOURCE NOTEBOOKS - Change this line when the repository has moved.
PROJECT_ROOT = Path("/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC")

DATA_ROOT = PROJECT_ROOT / "data"

PATHS = {
    # --------------------------------------------------------
    # Qiu developmental atlas: original inputs
    # --------------------------------------------------------
    "dev_shard_1":
        DATA_ROOT / "development/adata_JAX_dataset_1.h5ad",
    "dev_shard_2":
        DATA_ROOT / "development/adata_JAX_dataset_2.h5ad",
    "dev_shard_3":
        DATA_ROOT / "development/adata_JAX_dataset_3.h5ad",
    "dev_shard_4":
        DATA_ROOT / "development/adata_JAX_dataset_4.h5ad",

    "dev_cell_metadata":
        DATA_ROOT / "development/df_cell.csv",

    "dev_gene_metadata":
        DATA_ROOT / "development/df_gene.csv",

    # --------------------------------------------------------
    # Developmental objects created by Data_assembly.ipynb
    # --------------------------------------------------------
    "dev_selected_unbalanced":
        DATA_ROOT / "development/adata_dev_endoderm.h5ad",

    "dev_balanced":
        DATA_ROOT / "development/adata_dev_endoderm_balanced.h5ad",

    # --------------------------------------------------------
    # Burdziak progression cohort
    # --------------------------------------------------------
    "cancer_source":
        DATA_ROOT / "pdac/GSE207938_ProgressionCohort.h5ad",

    "cancer_full":
        DATA_ROOT / "pdac/burdziak_epifilterout_kept_N1_N2.h5ad",

    # --------------------------------------------------------
    # GRCm39 / Ensembl 111 reference used for gene translation
    # --------------------------------------------------------
    "gtf":
        DATA_ROOT / "ref/Mus_musculus.GRCm39.111.gtf",

    # --------------------------------------------------------
    # Aligned objects created by Data_assembly2.ipynb
    # --------------------------------------------------------
    "dev_aligned":
        DATA_ROOT / "development/A_dev_aligned.h5ad",

    "cancer_aligned":
        DATA_ROOT / "pdac/A_cancer_aligned.h5ad",

    "alignment_outdir":
        DATA_ROOT / "out/compare_dev_cancer",

    # --------------------------------------------------------
    # BERT training outputs
    # --------------------------------------------------------
    "bert_training_outdir":
        DATA_ROOT / "out/bert_dev_cancer_last_norm_3",

    "model_dir":
        DATA_ROOT / "models/bert_dev_learned",
}

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print()

path_table = pd.DataFrame({
    "key": list(PATHS.keys()),
    "path": [str(p) for p in PATHS.values()],
    "exists": [p.exists() for p in PATHS.values()],
})

path_table

PROJECT_ROOT = /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC
DATA_ROOT    = /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data



,key,path,exists
0,dev_shard_1,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
1,dev_shard_2,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
2,dev_shard_3,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
3,dev_shard_4,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
4,dev_cell_metadata,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
5,dev_gene_metadata,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
6,dev_selected_unbalanced,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
7,dev_balanced,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
8,cancer_source,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True
9,cancer_full,/home/jfpoyatos/Work/MNCN_24_25/Development_an...,True


In [3]:
AUDIT_COUNTS = {
    "development_cells": 34760,
    "cancer_cells": 28131,
    "total_mlm_cells": 62891,
    "shared_genes": 15376,
    "unintended_conjunctival_cells": 85,
}
AUDIT_COUNTS

{'development_cells': 34760,
 'cancer_cells': 28131,
 'total_mlm_cells': 62891,
 'shared_genes': 15376,
 'unintended_conjunctival_cells': 85}

### Path source from the original notebooks

Locations explicitly used in the original notebooks:

- `Data_assembly.ipynb` sets  
  `INDIRdev = "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/development"`
  and writes `adata_dev_endoderm_balanced.h5ad` there.
- The same notebook sets  
  `INDIRcan = "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/pdac"`
  and reads `GSE207938_ProgressionCohort.h5ad`.
- `Data_assembly2.ipynb` uses  
  `commonpath = "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/"`
  and writes `development/A_dev_aligned.h5ad` and
  `pdac/A_cancer_aligned.h5ad`.
- `Bert.ipynb` uses the same `commonpath` and saves the trained model and
  serialized token arrays under `data/models/bert_dev_learned/`.
- `Results_Section1` and `Results_Section2` load the same model directory and
  aligned objects, confirming the same path chain was used downstream.

## 2. Developmental atlas construction

Final balanced `dev_state_union` counts:

| State | Cells |
|---|---:|
| gut_epithelium_distal | 4,000 |
| ductal | 4,000 |
| lung_progenitor | 4,000 |
| lung_airway | 4,000 |
| gut_epithelium_proximal_or_mixed | 4,000 |
| liver_hepatocyte | 4,000 |
| foregut_other | 3,948 |
| acinar_or_tip | 3,747 |
| biliary | 1,742 |
| non_epithelial_endocrine | 1,323 |
| **Total** | **34,760** |

Important original mappings:
- `Pancreatic acinar cells` → `acinar_or_tip`
- `Foregut epithelial cells` → `ductal` / `pancreas_ductal_proxy`
- `Pancreatic islets` → `non_epithelial_endocrine`
- thyroid/parathyroid/thymic epithelium → `foregut_other`

The source metadata retain developmental day, `embryo_id`, experimental batch, major trajectory, and updated cell-type annotation.

### Recompute the developmental counts from the saved balanced object

In [4]:
if PATHS["dev_balanced"].exists():
    import scanpy as sc

    adata_dev_bal = sc.read_h5ad(PATHS["dev_balanced"])

    print("Balanced developmental object:", adata_dev_bal.shape)
    print()
    print("dev_state_union counts:")
    print(adata_dev_bal.obs["dev_state_union"].value_counts())
    print()
    print("Total cells:", adata_dev_bal.n_obs)

    assert adata_dev_bal.n_obs == 34760
else:
    print("Balanced developmental h5ad not found in candidate paths.")

Balanced developmental object: (34760, 24552)

dev_state_union counts:
dev_state_union
gut_epithelium_distal               4000
ductal                              4000
lung_progenitor                     4000
lung_airway                         4000
gut_epithelium_proximal_or_mixed    4000
liver_hepatocyte                    4000
foregut_other                       3948
acinar_or_tip                       3747
biliary                             1742
non_epithelial_endocrine            1323
Name: count, dtype: int64

Total cells: 34760


### Verify the unintended conjunctival-goblet inclusion

This reproduces the contamination count directly from the balanced developmental metadata rather than relying on the manually recorded value.

In [5]:
if PATHS["dev_balanced"].exists():
    celltype = adata_dev_bal.obs["celltype_update"].astype(str)
    state = adata_dev_bal.obs["dev_state_union"].astype(str)

    n_conj = int((celltype == "Conjunctival goblet cells").sum())
    n_conj_distal = int(
        ((celltype == "Conjunctival goblet cells") &
         (state == "gut_epithelium_distal")).sum()
    )

    print("Conjunctival goblet cells in balanced DEV:", n_conj)
    print("...assigned to gut_epithelium_distal:", n_conj_distal)
    print("Fraction of developmental corpus:", n_conj / adata_dev_bal.n_obs)
    print(
        "Fraction of distal-gut state:",
        n_conj_distal / (state == "gut_epithelium_distal").sum()
    )

    assert n_conj == 85
else:
    print("Balanced developmental h5ad not found.")

Conjunctival goblet cells in balanced DEV: 85
...assigned to gut_epithelium_distal: 85
Fraction of developmental corpus: 0.0024453394706559265
Fraction of distal-gut state: 0.02125


### Verify what `foregut_other` actually contains

In [6]:
if PATHS["dev_balanced"].exists():
    fg_other = adata_dev_bal[
        adata_dev_bal.obs["dev_state_union"].astype(str) == "foregut_other"
    ]

    print(fg_other.obs["celltype_update"].astype(str).value_counts())
else:
    print("Balanced developmental h5ad not found.")

celltype_update
Thymic epithelial cells         1937
Thyroid gland cells             1578
Parathyroid epithelial cells     433
Name: count, dtype: int64


In [7]:
expected_dev_counts = pd.Series({
    "gut_epithelium_distal": 4000,
    "ductal": 4000,
    "lung_progenitor": 4000,
    "lung_airway": 4000,
    "gut_epithelium_proximal_or_mixed": 4000,
    "liver_hepatocyte": 4000,
    "foregut_other": 3948,
    "acinar_or_tip": 3747,
    "biliary": 1742,
    "non_epithelial_endocrine": 1323,
}, name="expected_cells")

expected_dev_counts.to_frame()

,expected_cells
gut_epithelium_distal,4000
ductal,4000
lung_progenitor,4000
lung_airway,4000
gut_epithelium_proximal_or_mixed,4000
liver_hepatocyte,4000
foregut_other,3948
acinar_or_tip,3747
biliary,1742
non_epithelial_endocrine,1323


## 3. Cancer atlas construction and epithelial filtering

The Burdziak object used by the original pipeline contained **28,131 cells × 16,826 genes** with metadata including `batch`, `condition`, and `cluster`.

The original notebook explored an epithelial-expression score. A threshold retaining the top 70% would have yielded **19,692 cells**, but this gate was **not applied** to the MLM input. Instead:

```python
adata_epifilterout = adata_cancer.copy()
```

Thus **all 28,131 processed Burdziak cells** entered the downstream alignment and MLM pipeline.

The source study generated the cohort from lineage-traced pancreatic epithelial cells isolated experimentally by FACS, so the input was already epithelial-enriched upstream.

### Derived cancer states

| Conditions | State | Cells |
|---|---|---:|
| N1, N2 | ductal | 5,626 |
| K1, K1.5, K2, K3, K4 | transitional | 13,650 |
| K5, K6 | malignant_ductal | 8,855 |

These labels were downstream annotations, not MLM prediction targets.

### Recompute cancer cohort size and coarse-state counts

The saved `burdziak_epifilterout_kept_N1_N2.h5ad` file is the object that was actually retained after deciding **not** to apply the exploratory epithelial-score threshold.

In [8]:
if PATHS["cancer_full"].exists():
    import scanpy as sc

    adata_cancer_audit = sc.read_h5ad(PATHS["cancer_full"])

    print("Cancer object:", adata_cancer_audit.shape)
    print("obs columns:", list(adata_cancer_audit.obs.columns))
    print()

    if "cancer_state" in adata_cancer_audit.obs:
        print("Cancer-state counts:")
        print(adata_cancer_audit.obs["cancer_state"].value_counts())

    assert adata_cancer_audit.n_obs == 28131
else:
    print("Cancer h5ad not found in candidate paths.")

Cancer object: (28131, 16826)
obs columns: ['batch', 'condition', 'cluster', 'epi_score', 'cancer_state', 'condition_collapsed']

Cancer-state counts:
cancer_state
transitional        13650
malignant_ductal     8855
ductal               5626
Name: count, dtype: int64


### Reconstruct the derived cancer-state mapping from `condition`

This reproduces the three-state labels without relying on the precomputed `cancer_state` column.

In [9]:
COND2STATE = {
    "N1": "ductal",
    "N2": "ductal",
    "K1": "transitional",
    "K1.5": "transitional",
    "K2": "transitional",
    "K3": "transitional",
    "K4": "transitional",
    "K5": "malignant_ductal",
    "K6": "malignant_ductal",
}

if PATHS["cancer_full"].exists():
    reconstructed = (
        adata_cancer_audit.obs["condition"]
        .astype(str)
        .map(COND2STATE)
    )

    print(reconstructed.value_counts(dropna=False))

    expected = {
        "ductal": 5626,
        "transitional": 13650,
        "malignant_ductal": 8855,
    }

    for k, v in expected.items():
        assert int((reconstructed == k).sum()) == v
else:
    print("Cancer h5ad not found.")

condition
transitional        13650
malignant_ductal     8855
ductal               5626
Name: count, dtype: int64


### Show that the epithelial score was diagnostic rather than the final gate

If the retained object contains `epi_score`, this cell reports its distribution but confirms that the retained object still contains all 28,131 cells.

In [10]:
if PATHS["cancer_full"].exists():
    if "epi_score" in adata_cancer_audit.obs:
        print(adata_cancer_audit.obs["epi_score"].describe())
        print()
        print("Cells retained despite exploratory epi_score:", adata_cancer_audit.n_obs)
    else:
        print("epi_score not present in the saved object.")
else:
    print("Cancer h5ad not found.")

count    28131.000000
mean        16.070864
std         15.601848
min        -12.155810
25%          4.030074
50%         14.443037
75%         24.919104
max        114.292354
Name: epi_score, dtype: float64

Cells retained despite exploratory epi_score: 28131


In [11]:
expected_cancer_counts = pd.Series({
    "ductal": 5626,
    "transitional": 13650,
    "malignant_ductal": 8855,
}, name="expected_cells")

expected_cancer_counts.to_frame()

,expected_cells
ductal,5626
transitional,13650
malignant_ductal,8855


## 4. Gene harmonization

Audit summary:
- Developmental genes before alignment: **24,552**
- Cancer genes before mapping: **16,826**
- Directly mapped through GTF: **15,676**
- Mapped after external rescue: **15,950**
- Final unmapped cancer symbols: **876**
- Duplicate Ensembl mappings encountered: **4**
- Cancer genes after duplicate collapse: **15,946**
- Final shared genes: **15,376**

Final aligned matrices:
- Development: **34,760 × 15,376**
- Cancer: **28,131 × 15,376**

### Internal reproducibility note
The alignment call supplied `duplicate_agg="sum"`, but the implementation reported that duplicate genes were handled by **keeping the first occurrence and dropping the rest**. This is recorded for code cleanup before the final definitive rerun.

### Verify the final aligned matrix dimensions and shared vocabulary

In [12]:
if PATHS["dev_aligned"].exists() and PATHS["cancer_aligned"].exists():
    import scanpy as sc

    A_dev_aln_audit = sc.read_h5ad(PATHS["dev_aligned"])
    A_cancer_aln_audit = sc.read_h5ad(PATHS["cancer_aligned"])

    print("Development aligned:", A_dev_aln_audit.shape)
    print("Cancer aligned:", A_cancer_aln_audit.shape)
    print(
        "Identical gene order:",
        np.array_equal(
            A_dev_aln_audit.var_names.to_numpy(),
            A_cancer_aln_audit.var_names.to_numpy()
        )
    )

    assert A_dev_aln_audit.shape == (34760, 15376)
    assert A_cancer_aln_audit.shape == (28131, 15376)
    assert np.array_equal(
        A_dev_aln_audit.var_names.to_numpy(),
        A_cancer_aln_audit.var_names.to_numpy()
    )
else:
    print(
        "Aligned h5ad files not found. "
        "Add their local paths to PATHS['dev_aligned'] / PATHS['cancer_aligned']."
    )

Development aligned: (34760, 15376)
Cancer aligned: (28131, 15376)
Identical gene order: True


## 5. MLM featurization

For each aligned cell:

1. Counts were library-normalized.
2. A developmental gene-specific **nonzero median** was used as the reference baseline.
3. Expression was converted to developmental-relative values.
4. Relative values were transformed with `log1p`.
5. Genes lacking a developmental baseline were suppressed.
6. The **128 highest-scoring genes** were retained as tokens.

Sequence before dataset token:

```text
[CLS] <128 gene tokens> [SEP]
```

Actual sequence used:

```text
[DOM] [CLS] <128 gene tokens> [SEP]
```

where `[DOM]` is `[DEV]` or `[CANCER]`.

**Sequence length: 131 tokens per cell.**

### Recompute joint corpus size from the aligned objects

In [13]:
if PATHS["dev_aligned"].exists() and PATHS["cancer_aligned"].exists():
    n_dev = A_dev_aln_audit.n_obs
    n_can = A_cancer_aln_audit.n_obs
    n_total = n_dev + n_can

    print("Development:", n_dev)
    print("Cancer:", n_can)
    print("Joint MLM corpus:", n_total)

    assert n_total == 62891
else:
    print("Aligned objects not loaded.")

Development: 34760
Cancer: 28131
Joint MLM corpus: 62891


### Inspect the serialized MLM token arrays, if available

The original BERT notebook saved `X_tokens.npy`, `X_attn.npy`, and `domain.npy` in the model directory.

In [14]:
if PATHS["model_dir"].exists():
    model_dir = Path(PATHS["model_dir"])

    tok_path = model_dir / "X_tokens.npy"
    attn_path = model_dir / "X_attn.npy"
    dom_path = model_dir / "domain.npy"

    if tok_path.exists():
        X_tokens_audit = np.load(tok_path, mmap_mode="r")
        print("X_tokens:", X_tokens_audit.shape)
        print("Sequence length:", X_tokens_audit.shape[1])

        assert X_tokens_audit.shape[0] == 62891
        assert X_tokens_audit.shape[1] == 131
    else:
        print("X_tokens.npy not found:", tok_path)

    if dom_path.exists():
        domain_audit = np.load(dom_path, mmap_mode="r")
        vals, counts = np.unique(domain_audit, return_counts=True)
        print("Domain counts:", dict(zip(vals.tolist(), counts.tolist())))
    else:
        print("domain.npy not found:", dom_path)
else:
    print("Model directory not found.")

X_tokens: (62891, 131)
Sequence length: 131
Domain counts: {0: 34760, 1: 28131}


## 6. MLM masking and architecture

### Masking
- masking probability: **0.15**
- PAD/CLS/SEP/MASK/domain tokens excluded from masking
- among selected gene tokens:
  - ~80% → `[MASK]`
  - ~10% → random replacement
  - ~10% unchanged

### Audit note
Random replacements were sampled from token IDs beginning at 4 through the full vocabulary. Because the domain tokens were appended to that vocabulary, a random replacement could in principle draw a domain token. This should be cleaned in a definitive retraining.

### Architecture/training

| Parameter | Value |
|---|---:|
| hidden size | 256 |
| hidden layers | 6 |
| attention heads | 8 |
| intermediate size | 512 |
| top genes/cell | 128 |
| masking probability | 0.15 |
| epochs | 10 |
| micro-batch size | 8 |
| gradient accumulation | 4 |
| effective batch size | 32 |
| learning rate | 5e-4 |
| weight decay | 0.01 |
| warmup ratio | 0.06 |
| scheduler | linear |
| max gradient norm | 1.0 |
| fp16 | True |
| `dataloader_drop_last` | True |

The original model was trained on the joint corpus without a held-out validation split.

### Inspect the saved BERT configuration and vocabulary metadata

In [15]:
if PATHS["model_dir"].exists():
    import json

    model_dir = Path(PATHS["model_dir"])
    config_path = model_dir / "config.json"
    vocab_path = model_dir / "vocab_meta.json"

    if config_path.exists():
        config = json.loads(config_path.read_text())
        for k in [
            "vocab_size",
            "hidden_size",
            "num_hidden_layers",
            "num_attention_heads",
            "intermediate_size",
            "max_position_embeddings",
            "pad_token_id",
        ]:
            print(k, "=", config.get(k))
    else:
        print("config.json not found.")

    if vocab_path.exists():
        vocab_meta = json.loads(vocab_path.read_text())
        print("\nVocabulary metadata:")
        for k in [
            "PAD_ID", "CLS_ID", "SEP_ID", "MASK_ID",
            "DEV_DOM_ID", "CANCER_DOM_ID", "vocab_size"
        ]:
            print(k, "=", vocab_meta.get(k))
    else:
        print("vocab_meta.json not found.")
else:
    print("Model directory not found.")

vocab_size = 15382
hidden_size = 256
num_hidden_layers = 6
num_attention_heads = 8
intermediate_size = 512
max_position_embeddings = 133
pad_token_id = 0

Vocabulary metadata:
PAD_ID = 0
CLS_ID = 1
SEP_ID = 2
MASK_ID = 3
DEV_DOM_ID = 15380
CANCER_DOM_ID = 15381
vocab_size = 15382


## 7. Compact cell-flow table

| Stage | Development | Cancer |
|---|---:|---:|
| Source | Qiu prenatal mouse atlas | Burdziak progression cohort |
| Biological input | selected endodermal/epithelial states + pancreatic endocrine reference | lineage-traced pancreatic epithelial cohort |
| Additional computational epithelial gate | developmental cell-type selection | none applied |
| Balancing | max 4,000/state, seed 13 | none |
| Cells entering alignment | 34,760 | 28,131 |
| Shared genes entering MLM | 15,376 | 15,376 |
| MLM domain token | DEV | CANCER |
| Cells entering joint MLM | 34,760 | 28,131 |

**Total MLM corpus: 62,891 cells**

## 8. Revision finding A --> unintended conjunctival goblet cells

**85 conjunctival goblet cells** entered the balanced developmental corpus.

Scale:
- 85 / 34,760 developmental cells = **0.24%**
- 85 / 62,891 total MLM cells = **0.14%**

A downstream sensitivity analysis removed these 85 cells after training, refit the developmental PCA, and repeated the corrected geometry.

Observed effects:
- 75-PC variance explained: **0.981794 → 0.981805**
- mean squared old-vs-clean subspace overlap: **0.9999657**
- maximum principal angle: **2.71°**
- corrected-axis rotations:
  - late foregut: **0.047°**
  - acinar: **0.063°**
  - biliary: **0.025°**
- downstream cancer metrics changed by roughly **10⁻⁴**.

### Interpretation
This is a genuine coding error on the original pipeline but has negligible measurable downstream influence. The inclusion rule should be corrected before the final retraining, but repeated retraining during AE1 is unnecessary.

## 9. Revision finding B —-> original “foregut” root

The developmental assembly mapped actual Qiu `Foregut epithelial cells` to a pancreatic ductal-proxy category, while `foregut_other` contained thyroid, thymic, and parathyroid epithelial populations.

The original downstream developmental-direction analysis later used `foregut_other` as the shared “foregut” root.

### Interpretation
This is a **substantive biological-definition problem**, but correction belongs to **AE3: refine the developmental biology**, not AE1.

Preliminary sensitivity analyses already showed that replacing this root materially rotates the developmental axes, so it must be corrected before final interpretation.

## 10. Revision finding C —> problem with masking

The intended MLM procedure is standard BERT-like corruption. For about 15% of gene positions, the model is asked to predict the original gene. Of those selected positions, ~80% are replaced by [MASK], ~10% by a random token, and ~10% are left unchanged. The notebook correctly prevents special/domain tokens themselves from being selected for masking

rand_ids = torch.randint(low=4, high=vocab_size, ...)
Because vocab_size = 15382, this samples IDs from 4 through 15381. That range contains not only all gene IDs but also the two domain IDs at the end:
genes:       4 ... 15379
DEV token:       15380
CANCER token:    15381

So suppose a cell contains a gene token such as Krt19, and that position is selected for the “replace with a random token” part of MLM corruption. Almost always it gets another gene. But very rarely it could become:
... Krt8  Pdx1  [CANCER]  Sox9 ...
even though that position is supposed to contain a gene, not a domain label.

### Interpretation
This is not a serious problem for the existing results, but it will be fixed after the definitive retraining.
The probability of the above to happen was tiny. 


## 11. AE1 final closure statement

AE1 does **not** show that the original MLM was fundamentally invalid. It shows why a definitive paired retraining is scientifically cleaner for the revision.

The original pipeline used 34,760 developmental cells and 28,131 Burdziak cells over 15,376 shared genes. The audit identified: (i) 85 unintended conjunctival goblet cells introduced through a broad substring mapping, with negligible downstream sensitivity in the old embedding; (ii) four cancer gene symbols mapped to Ensembl IDs already assigned to another symbol. The original helper intentionally retained the first occurrence and discarded subsequent aliases. Although the higher-level function retained a legacy duplicate_agg="sum" argument, aggregation was not performed. We therefore treat this as an interface/documentation inconsistency rather than a substantive pipeline error. (iii) rare random MLM replacements that could draw `[DEV]` or `[CANCER]`; and (iv) a separate downstream biological-definition problem in which thyroid/thymic/parathyroid epithelium was used as a pancreatic “foregut” root.

AE1b corrects the first three implementation issues and retrains paired joint/development-only encoders. The fourth issue remains a **AE3 biological-reference issue**: the revised developmental directions are defined after embedding and are not trained into the MLM.



### Derivation of this notebook

Every headline number above is either:

1. recomputed directly from the serialized objects produced by the original pipeline; or  
2. explicitly labeled as a revised result obtained from the original source notebooks when the intermediate object itself was not serialized.
